# ⚡ LendingClub End-to-End Default Prediction Pipeline
## Enterprise Executive Presentation: Automated PySpark Credit Scoring & Portfolio Risk Benchmarking

---

### 💼 Executive Overview & Business Objectives
Modern retail credit platforms require scalable, automated underwriting pipelines to evaluate borrower risk, reduce credit default losses, and optimize **Risk-Adjusted Return on Capital (RAROC)**.

This pipeline delivers an enterprise-grade, end-to-end **Apache Spark (PySpark)** data engineering and machine learning system processing **2.2+ Million** historical LendingClub loan applications.

#### 🎯 Strategic Technical & Financial Goals:
1. **High-Throughput Distributed Data Pipeline**: Automate data ingestion, schema selection, and null cleansing across multi-million row datasets.
2. **Automated Feature Transformation & Risk Flagging**: Standardize loan terms, encode borrower income verification, cap delinquency outliers, and encode credit grade ordinals.
3. **Class Imbalance Resolution**: Apply stratified downsampling to majority performing loans (`Fully Paid`) while preserving 100% of default write-offs (`Charged Off / Default`) and early warning delinquencies (`Late / Grace Period`).
4. **Vectorized Multi-Model Benchmarking**: Compare **Logistic Regression** (Regulatory Compliance Scorecard), **Random Forest** (Non-linear Risk Interaction), and **Multilayer Perceptron** (Deep Representation Network).

---

In [10]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, when
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd

# Spark Engine Setup with Interactive Eager Notebook Evaluation enabled
spark = SparkSession.builder \
    .appName("lending-club") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .config("spark.sql.repl.eagerEval.maxNumRows", 10) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.ansi.enabled", "false")

# 1. Enterprise Data Ingestion & Quality Control Protocol

---

## 📥 Distributed Data Loading & Schema Filtering
We load raw LendingClub credit records directly into a PySpark DataFrame, restricting schema ingestion to 23 core underwriting parameters and purging incomplete records across the distributed cluster.

### 📋 Ingestion Parameters:
* **Borrower Credit Metrics**: `annual_inc`, `dti` (Debt-to-Income), `last_fico_range_high`, `last_fico_range_low`.
* **Loan Financial Terms**: `loan_amnt`, `installment`, `int_rate`, `term`, `grade`.
* **Delinquency & Recovery History**: `acc_now_delinq`, `delinq_amnt`, `tax_liens`, `recoveries`, `collection_recovery_fee`.
* **Demographic & Structure Flags**: `purpose`, `addr_state`, `home_ownership`, `verification_status`, `application_type`, `initial_list_status`.

In [11]:
# 💻 Code (Ingestion & Quality Cleanse)
selected_columns = [
    "id", "purpose", "term", "verification_status", "acc_now_delinq", "addr_state", 
    "annual_inc", "application_type", "dti", "grade", "home_ownership", "initial_list_status", 
    "installment", "int_rate", "loan_amnt", "loan_status", "tax_liens", "delinq_amnt", 
    "policy_code", "last_fico_range_high", "last_fico_range_low", "recoveries", "collection_recovery_fee"
]

path = '/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz'

# Load with native type inference and instantly drop missing blocks
df = spark.read.csv(path, header=True, inferSchema=True).select(selected_columns).na.drop()

# 2. Feature Engineering & Portfolio Class Imbalance Resolution

---

## 🛠️ Data Transformations & Stratified Downsampling Strategy

### 🔧 Feature Transformation Logic:
1. **Tenure Extraction (`term`)**: Regex extraction converting string loan duration (`36 months`, `60 months`) into quantitative integers (`36`, `60`).
2. **Verification Flag (`verification_status_encoded`)**: Binary risk flag (`0` = Verified / Source Verified, `1` = Not Verified).
3. **Delinquency Capping (`acc_now_delinq`)**: Capping active delinquency accounts at **`4`** to prevent gradient instability during model fitting.
4. **Applicant Structure (`application_type`)**: Mapping Joint Applications (`0`) and Individual Applications (`1`).

---

### ⚖️ Multi-Class Target Mapping & Controlled Downsampling:
* **`Class 0: Fully Paid`**: Performing loans (Principal + Interest recovered).
* **`Class 1: Late (16–120 days) / Grace Period`**: Early warning delinquency signals.
* **`Class 2: Charged Off / Default`**: Severe credit loss write-offs.
* **Downsampling Execution**: Class 0 is sampled down to **30%**, while **100% of Class 1 and Class 2 loss events are preserved** to eliminate majority-class model bias.

In [12]:
# 💻 Code (Transformations & Stratified Downsampling)

# 1. Feature transformations & Target Mapping
df_prep = (
    df.withColumn("term", regexp_extract("term", r"\d+", 0).cast("int"))
      .withColumn("verification_status_encoded", when(col("verification_status").isin("Verified", "Source Verified"), 0).otherwise(1))
      .withColumn("acc_now_delinq", when(col("acc_now_delinq").cast("int") >= 4, 4).otherwise(col("acc_now_delinq").cast("int")))
      .withColumn("application_type", when(col("application_type") == "Joint App", 0).when(col("application_type") == "Individual", 1))
      .withColumn("target", when(col("loan_status") == "Fully Paid", 0)
                           .when(col("loan_status").isin("Late (16-30 days)", "Late (31-120 days)", "In Grace Period"), 1)
                           .when(col("loan_status").isin("Charged Off", "Default"), 2))
      .drop("verification_status", "loan_status", "id")
      .filter(col("acc_now_delinq").between(0, 4) & col("application_type").isNotNull() & col("target").isNotNull())
)

# 2. Downsample Class 0 to 30% & drop NaNs
df_sampled = df_prep.sampleBy("target", fractions={0: 0.3, 1: 1.0, 2: 1.0}, seed=42).na.drop()

# 3. High-Speed Matrix Vectorization & Feature Normalization

---

## 🧬 Parallel Feature Indexing, Vector Assembly & Scaling

### ⚡ Vector Pipeline Execution:
1. **Categorical Encoding (`StringIndexer` + `OneHotEncoder`)**: Converts nominal categorical variables (`purpose`, `addr_state`, `home_ownership`, `initial_list_status`, `grade`) into numerical vectors without unrolling slow single-column loops.
2. **PySpark `VectorAssembler`**: Assembles all processed numerical and one-hot vector features into a consolidated high-dimensional input vector (`raw_features`).
3. **Data Partitioning (80/10/10 Split)**: Stratified partitioning into Training (80%), Validation (10%), and Out-of-Sample Test (10%) sets.
4. **Data-Leakage Prevention (`MinMaxScaler`)**: `MinMaxScaler` is fit **strictly on the Training split** and applied across Validation and Test splits to scale values into $[0, 1]$.

In [13]:
# 💻 Code (Parallel Indexing, Casting, and Splitting)

# 1. Categorical Encoding (StringIndexer + OneHotEncoder)
cat_cols = [c for c in ['purpose', 'addr_state', 'home_ownership', 'initial_list_status', 'grade'] if c in df_sampled.columns]
idx_cols, vec_cols = [f"{c}_idx" for c in cat_cols], [f"{c}_vec" for c in cat_cols]

df_prep = StringIndexer(inputCols=cat_cols, outputCols=idx_cols).fit(df_sampled).transform(df_sampled)
df_prep = OneHotEncoder(inputCols=idx_cols, outputCols=vec_cols).fit(df_prep).transform(df_prep).drop(*cat_cols, *idx_cols)

# 2. Dynamic Type Casting & Vector Assembly
features = [c for c in df_prep.columns if c != "target"]
df_typed = df_prep.select([col(c).cast("double") if not c.endswith("_vec") else col(c) for c in df_prep.columns])
df_vector = VectorAssembler(inputCols=features, outputCol="raw_features").transform(df_typed)

# 3. 80/10/10 Split & Scaler Fit on Train Only
train, temp = df_vector.randomSplit([0.8, 0.2], seed=42)
val, test = temp.randomSplit([0.5, 0.5], seed=42)

scaler = MinMaxScaler(inputCol="raw_features", outputCol="features").fit(train)
train_scaled, val_scaled, test_scaled = scaler.transform(train), scaler.transform(val), scaler.transform(test)

# 4. Multi-Model Machine Learning Evaluation & Strategic Recommendations

---

## 🤖 Multi-Classifier Training & Metric Comparison Suite
We execute parallel training across three distinct classifier architectures, calculating Accuracy and weighted F1-Scores across Train, Validation, and Test splits.

---

### 📊 Model Performance & Architectural Trade-offs:
* **Logistic Regression**: Linear benchmark delivering transparent log-odds coefficients required for FCRA/ECOA regulatory compliance scorecards.
* **Random Forest**: Ensemble decision tree capturing non-linear compounding risk factors (e.g., high DTI combined with low FICO scores).
* **Multilayer Perceptron (MLP)**: Deep learning network (`[Features, 64, 32, 3]`) performing non-linear feature abstraction across scaled input vectors.

---

### 🎯 Key Executive Recommendations for Deployment:

1. **Dynamic Risk-Based Pricing**:
   - Feed model-predicted default probabilities ($P(\text{Default})$) into automated pricing engines to dynamically adjust interest rate risk premiums.

2. **Early Warning System (EWS)**:
   - Utilize Class 1 predictions (`Late / Grace Period`) to trigger automated borrower outreach, payment restructuring, or secondary market loan sales before charge-off.

3. **Loss Matrix & RAROC Optimization**:
   - Align underwriting approval cutoffs with financial loss matrices: avoiding a single default saves 100% principal, whereas rejecting a good loan forfeits only marginal interest income.

In [14]:
# 💻 Code (Model Training & Metric Comparison Layout)
# 1. Pre-load data splits directly into RAM to prevent evaluation pipeline lag
for ds in [train_scaled, val_scaled, test_scaled]: ds.cache().count()

# 2. Shared Evaluator instance (reused to avoid unnecessary object creation)
evaluator = MulticlassClassificationEvaluator(labelCol="target")

# 3. Model definitions (auto-scaled MLP input layers based on vector length)
models = [
    (LogisticRegression(featuresCol="features", labelCol="target"), "Logistic Regression"),
    (RandomForestClassifier(featuresCol="features", labelCol="target"), "Random Forest"),
    (MultilayerPerceptronClassifier(layers=[train_scaled.select("features").head()[0].size, 64, 32, 3], 
                                    featuresCol="features", labelCol="target", maxIter=100, seed=123), "Neural Network")
]

# 4. Sequential Fit & Single-Pass Evaluation
def evaluate(model, name):
    fit_m = model.fit(train_scaled)
    res = {"Model": name}
    for label, ds in [("Train", train_scaled), ("Validation", val_scaled), ("Test", test_scaled)]:
        preds = fit_m.transform(ds)  # Calculate predictions ONCE per split
        res[f"Accuracy ({label})"] = round(evaluator.setMetricName("accuracy").evaluate(preds), 3)
        res[f"F1 Score ({label})"] = round(evaluator.setMetricName("f1").evaluate(preds), 3)
    return res

# Render as a clean Pandas DataFrame table
pd.DataFrame([evaluate(m, name) for m, name in models])

,Model,Accuracy (Train),F1 Score (Train),Accuracy (Validation),F1 Score (Validation),Accuracy (Test),F1 Score (Test)
0,Logistic Regression,0.879,0.861,0.878,0.860,0.877,0.859
1,Random Forest,0.874,0.849,0.873,0.849,0.874,0.849
2,Neural Network,0.860,0.840,0.859,0.839,0.859,0.839
